# Fig.8-style combined deterioration: Q-ACCeSS-T throughput robustness

Session from `run_qaccess_t_combined_deterioration_eval.sh`:
- `combined_baseline` vs `combined_qaccess_t_dynamic`
- Same topology, input, timeout, and **combined** delay+loss profile on Path B (`h2-eth1`)

**Primary metric:** QUIC wire throughput on h1 captures — filter `udp`, **both directions**, epoch-aligned with per-session `global_t0`. Total = Path A + Path B.

**Explanatory metrics:** runtime samples (`owd_ms`, `loss_rate`, coeffs, gain/backoff). PCAP does not directly report configured netem delay/loss; use tc log + runtime samples for impairment evidence.

Vertical markers: deterioration **90s**, recovery **100s**.

> The 1% gate permits a controlled runtime update when the RF-predicted improvement exceeds 1%; the actual performance effect is validated using packet traces and runtime measurements.

In [ ]:
import json
import math
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

REPO = Path.cwd()
if not (REPO / "scripts" / "analyze").is_dir():
    REPO = Path.cwd().parent.parent
sys.path.insert(0, str(REPO / "scripts" / "analyze"))

from qaccess_impairment_eval_analyze import (  # noqa: E402
    IMPAIRMENT_WINDOWS,
    PRESETS,
    load_run_wire_timeseries,
    mean_in_window,
)

PATH_FILTER = "udp"
DETERIORATION_START = 90
RECOVERY_START = 100
BASELINE_DIR = PRESETS["combined"]["baseline_dir"]
DYNAMIC_DIR = PRESETS["combined"]["dynamic_dir"]
OUT = REPO / "derived" / "combined_deterioration_compare"
OUT.mkdir(parents=True, exist_ok=True)

In [ ]:
# Locate latest combined-deterioration session (override SESSION manually if needed)
SESSION = None
last = REPO / "logs_exp" / ".last_session"
if last.is_file():
    cand = REPO / last.read_text().strip()
    if cand.is_dir() and "combined_deterioration" in cand.name:
        SESSION = cand
if SESSION is None:
    hits = sorted((REPO / "logs_exp").glob("session_combined_deterioration_*"))
    SESSION = hits[-1] if hits else None
assert SESSION and SESSION.is_dir(), f"No session found under {REPO / 'logs_exp'}"

baseline_dir = SESSION / BASELINE_DIR
dynamic_dir = SESSION / DYNAMIC_DIR
print("SESSION:", SESSION)
print("baseline:", baseline_dir, "exists=", baseline_dir.is_dir())
print("dynamic:", dynamic_dir, "exists=", dynamic_dir.is_dir())

In [ ]:
ts_base = load_run_wire_timeseries(baseline_dir)
ts_dyn = load_run_wire_timeseries(dynamic_dir)
assert not ts_base.empty and not ts_dyn.empty, "Missing pcaps — check run folders"

ts_base.to_csv(OUT / "combined_wire_timeseries_baseline.csv", index=False)
ts_dyn.to_csv(OUT / "combined_wire_timeseries_dynamic.csv", index=False)
print(ts_base.tail(3))
print(ts_dyn.tail(3))

In [ ]:
rows = []
for label, ts in [("baseline", ts_base), ("qaccess_t_dynamic", ts_dyn)]:
    for wname, lo, hi in IMPAIRMENT_WINDOWS:
        total = mean_in_window(ts, "total_quic_wire_mbps", lo, hi)
        pa = mean_in_window(ts, "path_a_quic_wire_mbps", lo, hi)
        pb = mean_in_window(ts, "path_b_quic_wire_mbps", lo, hi)
        share_b = pb / total * 100 if total and total == total and total > 0 else float("nan")
        rows.append({
            "method": label,
            "window": wname,
            "t_lo": lo,
            "t_hi": hi,
            "total_quic_wire_mbps": total,
            "path_a_quic_wire_mbps": pa,
            "path_b_quic_wire_mbps": pb,
            "path_b_share_pct": share_b,
        })

df_win = pd.DataFrame(rows)
df_win.to_csv(OUT / "combined_throughput_windows.csv", index=False)

base = df_win[df_win["method"] == "baseline"].set_index("window")
dyn = df_win[df_win["method"] == "qaccess_t_dynamic"].set_index("window")
imp = []
for w in dyn.index:
    b = float(base.loc[w, "total_quic_wire_mbps"]) if w in base.index else float("nan")
    e = float(dyn.loc[w, "total_quic_wire_mbps"])
    imp.append({
        "window": w,
        "baseline_mbps": b,
        "dynamic_mbps": e,
        "improvement_pct": (e - b) / b * 100 if b == b and b > 0 and e == e else float("nan"),
    })
df_imp = pd.DataFrame(imp)
df_imp.to_csv(OUT / "combined_improvement_vs_baseline.csv", index=False)
display(df_win)
display(df_imp)

In [ ]:
def _mark_events(ax):
    ax.axvline(DETERIORATION_START, color="red", linestyle="--", alpha=0.7, label="deterioration 90s")
    ax.axvline(RECOVERY_START, color="green", linestyle="--", alpha=0.7, label="recovery 100s")
    ax.axvspan(DETERIORATION_START, RECOVERY_START, color="red", alpha=0.08)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(ts_base["time_s"], ts_base["total_quic_wire_mbps"], label="baseline")
ax.plot(ts_dyn["time_s"], ts_dyn["total_quic_wire_mbps"], label="qaccess_t_dynamic")
_mark_events(ax)
ax.set_title("Total QUIC wire throughput (udp, both directions)")
ax.set_xlabel("Time from global_t0 (s)")
ax.set_ylabel("Mbps")
ax.legend()
fig.tight_layout()
fig.savefig(OUT / "plot_total_throughput.png", dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(ts_dyn["time_s"], ts_dyn["path_a_quic_wire_mbps"], label="dynamic Path A")
ax.plot(ts_dyn["time_s"], ts_dyn["path_b_quic_wire_mbps"], label="dynamic Path B")
_mark_events(ax)
ax.set_title("Dynamic per-path QUIC wire throughput")
ax.set_xlabel("Time from global_t0 (s)")
ax.set_ylabel("Mbps")
ax.legend()
fig.tight_layout()
fig.savefig(OUT / "plot_dynamic_per_path.png", dpi=150)
plt.show()

In [ ]:
share = ts_dyn["path_b_quic_wire_mbps"] / ts_dyn["total_quic_wire_mbps"].replace(0, float("nan")) * 100
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(ts_dyn["time_s"], share, label="Path B share %")
_mark_events(ax)
ax.set_title("Dynamic Path B traffic share")
ax.set_xlabel("Time from global_t0 (s)")
ax.set_ylabel("Path B % of total bytes")
ax.legend()
fig.tight_layout()
fig.savefig(OUT / "plot_path_b_share.png", dpi=150)
plt.show()

In [ ]:
samples_path = dynamic_dir / "derived_snapshots" / "qaccess_runtime_samples.csv"
if not samples_path.is_file():
    samples_path = REPO / "derived" / "qaccess_runtime_samples.csv"
samples = pd.read_csv(samples_path) if samples_path.is_file() else pd.DataFrame()
print("runtime samples:", samples_path, "rows=", len(samples))

if not samples.empty and "owd_ms" in samples.columns:
    tcol = "t_s" if "t_s" in samples.columns else ("t" if "t" in samples.columns else None)
    if tcol:
        fig, ax = plt.subplots(figsize=(12, 4))
        for path, sub in samples.groupby(samples.get("path_id", samples.get("path", "all"))):
            ax.plot(sub[tcol], sub["owd_ms"], label=f"owd path {path}")
        _mark_events(ax)
        ax.set_title("Runtime owd_ms by path")
        ax.set_xlabel("time (s)")
        ax.set_ylabel("owd_ms")
        ax.legend()
        fig.tight_layout()
        fig.savefig(OUT / "plot_runtime_owd.png", dpi=150)
        plt.show()

    loss_col = "loss_rate" if "loss_rate" in samples.columns else None
    if loss_col and tcol:
        fig, ax = plt.subplots(figsize=(12, 4))
        for path, sub in samples.groupby(samples.get("path_id", samples.get("path", "all"))):
            ax.plot(sub[tcol], sub[loss_col], label=f"loss path {path}")
        _mark_events(ax)
        ax.set_title("Runtime loss signal by path")
        ax.set_xlabel("time (s)")
        ax.set_ylabel(loss_col)
        ax.legend()
        fig.tight_layout()
        fig.savefig(OUT / "plot_runtime_loss.png", dpi=150)
        plt.show()

    for cols, title, fname in [
        (["alpha", "beta", "gamma"], "Runtime coefficients", "plot_coeffs.png"),
        (["gain", "backoff"], "Runtime gain/backoff", "plot_gain_backoff.png"),
    ]:
        use = [c for c in cols if c in samples.columns]
        if use and tcol:
            fig, ax = plt.subplots(figsize=(12, 4))
            for c in use:
                ax.plot(samples[tcol], samples[c], label=c)
            _mark_events(ax)
            ax.set_title(title)
            ax.set_xlabel("time (s)")
            ax.legend()
            fig.tight_layout()
            fig.savefig(OUT / fname, dpi=150)
            plt.show()

In [ ]:
def _load_json(path):
    return json.loads(path.read_text()) if path.is_file() else {}

before = _load_json(SESSION / "combined_qaccess_t_dynamic_coeffs_before.json")
after = _load_json(SESSION / "combined_qaccess_t_dynamic_coeffs_after.json")
resp_path = dynamic_dir / "derived_snapshots" / "qaccess_update_response.json"
if not resp_path.is_file():
    resp_path = REPO / "derived" / "qaccess_update_response.json"
response = _load_json(resp_path)

print("coeffs before:", {k: before.get(k) for k in ("alpha", "beta", "gamma")})
print("coeffs after:", {k: after.get(k) for k in ("alpha", "beta", "gamma")})
print("worker response:", {k: response.get(k) for k in (
    "status", "skip_reason", "improvement_pct", "improvement_min_pct",
    "predicted_current_bw_bps", "predicted_best_bw_bps", "request_id"
)})

## Interpretation checklist

1. **90–100s throughput:** Did Q-ACCeSS-T maintain higher total throughput than baseline?
2. **Drop magnitude:** Was the deterioration-window drop smaller for dynamic?
3. **Recovery 100–110s:** Faster return toward pre-deterioration levels?
4. **Path shift:** Did Path B share fall during 90–100s?
5. **1% gate:** Did worker `status=ok` (coefficient update accepted)?
6. **Selected coeffs:** Compare before/after alpha/beta/gamma and worker response.
7. **Go reload:** Search dynamic pull log for `loaded coefficients` / reload timestamps if SAVE_LOGS=1.
8. **Real effect:** Compare throughput before vs after update window — prediction is necessary but not sufficient.
9. **Stability 150–200s:** Check post-update oscillation in total throughput or gain/backoff.